# LAB | Feature Engineering

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In [35]:
# Step 0 - Import libraries and dataset
# -------------------------------------

# Import the libraries needed for this section
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# Standard seed used throughout the notebook, for reproducibility
SEED = 1

# Load the Iris dataset as a DataFrame
iris_data = load_iris(as_frame=True)
iris_df = iris_data.data.copy()
iris_df['species'] = iris_data.target.map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})
iris_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   sepal length (cm)  150 non-null    float64
 1   sepal width (cm)   150 non-null    float64
 2   petal length (cm)  150 non-null    float64
 3   petal width (cm)   150 non-null    float64
 4   species            150 non-null    object 
dtypes: float64(4), object(1)
memory usage: 6.0+ KB


In [36]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


**Check the shape of your data**

In [37]:
spaceship.shape

(8693, 14)

**Check for data types**

In [38]:
spaceship.dtypes

PassengerId      object
HomePlanet       object
CryoSleep        object
Cabin            object
Destination      object
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name             object
Transported        bool
dtype: object

**Check for missing values**

In [39]:
spaceship.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

There are multiple strategies to handle missing data

- Removing all rows or all columns containing missing data.
- Filling all missing values with a value (mean in continouos or mode in categorical for example).
- Filling all missing values with an algorithm.

For this exercise, because we have such low amount of null values, we will drop rows containing any missing value. 

In [40]:
spaceship = spaceship.dropna()

- **Cabin** is too granular - transform it in order to obtain {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'T'}

In [41]:
spaceship["Cabin"].unique()

array(['B/0/P', 'F/0/S', 'A/0/S', ..., 'G/1499/S', 'G/1500/S', 'E/608/S'],
      shape=(5305,), dtype=object)

In [42]:
spaceship["Cabin"] = spaceship["Cabin"].str[0]

In [43]:
spaceship["Cabin"].unique()

array(['B', 'F', 'A', 'G', 'E', 'C', 'D', 'T'], dtype=object)

- Drop PassengerId and Name

In [44]:
spaceship = spaceship.drop(columns = ["PassengerId", "Name"])

- For non-numerical columns, do dummies.

In [56]:
categorical_cols = spaceship.select_dtypes(include='object').columns
spaceship_encoded = pd.get_dummies(spaceship, columns=categorical_cols, drop_first=True).astype(int)

In [57]:
spaceship_encoded.head()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
0,39,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0
1,24,109,9,25,549,44,1,0,0,0,0,0,0,0,1,0,0,0,1,0
2,58,43,3576,0,6715,49,0,1,0,0,0,0,0,0,0,0,0,0,1,1
3,33,0,1283,371,3329,193,0,1,0,0,0,0,0,0,0,0,0,0,1,0
4,16,303,70,151,565,2,1,0,0,0,0,0,0,0,1,0,0,0,1,0


**Perform Train Test Split**

In [65]:
X = spaceship_encoded.drop(columns=["Transported"])
y = spaceship_encoded["Transported"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y)  # Keeps the same proportion of the 3 species in both sets

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')

X_train shape: (5284, 19)
X_test shape:  (1322, 19)


**Model Selection**

In this exercise we will be using **KNN** as our predictive model.

In [66]:
scaler = StandardScaler()
scaler.set_output(transform="pandas")
# set_output(transform="pandas") makes the scaler return a DataFrame (keeps the real column names and index)

# Fit ONLY on X_train --> avoids leaking any information from the test set into the scaling parameters
X_train_scaled_df = scaler.fit_transform(X_train)

# Then apply that same transformation to X_test
X_test_scaled_df = scaler.transform(X_test)

# BEFORE: original scale
print('BEFORE scaling (X_train):')
print(X_train.describe().loc[['mean', 'std']].round(4))
print()

# AFTER: standardized scale
print('AFTER scaling (X_train):')
print(X_train_scaled_df.describe().loc[['mean', 'std']].round(4))

BEFORE scaling (X_train):
          Age  RoomService  FoodCourt  ShoppingMall        Spa     VRDeck  \
mean  28.9080     222.4037   475.6171      176.9722   308.2684   296.1684   
std   14.4869     639.4044  1620.5749      592.2009  1108.4575  1118.2181   

      HomePlanet_Europa  HomePlanet_Mars  CryoSleep_True  Cabin_B  Cabin_C  \
mean             0.2561           0.2087          0.3556   0.0973   0.0893   
std              0.4365           0.4064          0.4787   0.2964   0.2852   

      Cabin_D  Cabin_E  Cabin_F  Cabin_G  Cabin_T  Destination_PSO J318.5-22  \
mean   0.0590   0.1016   0.3231   0.2986   0.0004                      0.092   
std    0.2357   0.3022   0.4677   0.4577   0.0195                      0.289   

      Destination_TRAPPIST-1e  VIP_True  
mean                   0.6945    0.0244  
std                    0.4606    0.1543  

AFTER scaling (X_train):
         Age  RoomService  FoodCourt  ShoppingMall     Spa  VRDeck  \
mean -0.0000       0.0000     0.0000       -

In [67]:
knn_model = KNeighborsClassifier(n_neighbors=5, weights='distance')
knn_model.fit(X_train_scaled_df, y_train)  # Fit on train

print('Model trained!')
# KNN does NOT learn coefficients (no coef_) --> it just memorizes the training points and their distances!!!

Model trained!


- Evaluate your model's performance. Comment it

In [68]:


# Evaluate the model on both Train and Test, to check for overfitting
y_pred_train = knn_model.predict(X_train_scaled_df)  # train
y_pred_test = knn_model.predict(X_test_scaled_df)    # test

accuracy_train = accuracy_score(y_train, y_pred_train)  # train, just to compare with the report below
print(f'Accuracy (Train): {accuracy_train * 100:.2f}%')
print()
print(classification_report(y_test, y_pred_test))

Accuracy (Train): 94.00%

              precision    recall  f1-score   support

           0       0.77      0.76      0.76       656
           1       0.77      0.77      0.77       666

    accuracy                           0.77      1322
   macro avg       0.77      0.77      0.77      1322
weighted avg       0.77      0.77      0.77      1322



In [ ]:
#It's undeerfitting